# Pipeline: Modelo de Predicción y Visualización
## Caso Tipo B: IRA y Neumonía en menores de 5 años (Perú)

**Challenge 1 (Individual): Pipeline de Machine Learning, UPCH**

**Tarea de Machine Learning: Regresión (aprendizaje supervisado).**

**Objetivo:** predecir la **tasa trimestral de episodios de IRA por 1 000 menores de 5 años** de una región a partir de la cobertura de vacunación, el clima, la pobreza y la presencia de la campaña "Abrigamos vidas".

**Por qué hace falta un dataset sintético:** la infografía solo reporta 3 totales nacionales anuales y declara como **no reportados** la cobertura real de vacunación, la distribución regional de casos y los factores climáticos y socioeconómicos. Por eso se genera un dataset sintético **calibrado para que sus totales anuales coincidan con los del anexo** (1 003 178, 883 959 y 824 978 episodios). Las relaciones entre variables son **supuestos de simulación, no hallazgos epidemiológicos**.

## 1. Datos reales del caso (anexo)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

anios = [2023, 2024, 2025]
episodios_ira = {2023: 1003178, 2024: 883959, 2025: 824978}   # anexo
muertes_neumonia = {2023: 113, 2024: 82, 2025: 51}            # anexo

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar([str(a) for a in anios], [episodios_ira[a] for a in anios], color="#0E7C7B")
axes[0].set_title("Episodios de IRA (< 5 años)"); axes[0].set_ylabel("Episodios")
axes[1].bar([str(a) for a in anios], [muertes_neumonia[a] for a in anios], color="#B0201B")
axes[1].set_title("Defunciones por neumonía (< 5 años)"); axes[1].set_ylabel("Defunciones")
plt.tight_layout(); plt.show()

## 2. Generación del dataset sintético

Unidad de observación: **región × trimestre** (25 regiones × 12 trimestres = 300 filas).

| Variable | Rol | Descripción simulada |
|---|---|---|
| `cobertura_neumococo_pct` | entrada | Cobertura de vacuna antineumocócica (%); sube con los años |
| `cobertura_influenza_pct` | entrada | Cobertura de vacuna contra influenza (%) |
| `temperatura_media_c` | entrada | Temperatura media del trimestre (°C); menor en invierno y en regiones altas |
| `pobreza_pct` | entrada | Pobreza (%), factor socioeconómico |
| `campana_abrigamos_vidas` | entrada | 1 si la campaña estuvo activa en esa región-trimestre |
| `tasa_ira_x1000` | **objetivo** | Episodios de IRA por 1 000 menores de 5 años en el trimestre |

In [ ]:
rng = np.random.default_rng(42)
n_reg, n_trim = 25, 12
regiones = [f"Region_{i+1:02d}" for i in range(n_reg)]

# Caracteristicas fijas por region
altitud = rng.uniform(5, 4200, n_reg)                     # msnm
pobreza_base = rng.uniform(10, 55, n_reg)                 # %
pob_menor5 = rng.lognormal(mean=0, sigma=0.8, size=n_reg)
pob_menor5 = pob_menor5 / pob_menor5.sum() * 2_800_000    # poblacion < 5 anos total (supuesto)
cob_neumo_base = rng.uniform(55, 80, n_reg)
cob_flu_base = rng.uniform(40, 70, n_reg)

filas = []
for r in range(n_reg):
    for t in range(n_trim):
        anio = 2023 + t // 4
        trim = t % 4
        estacional = -3.5 * np.cos(2 * np.pi * (trim - 0.5) / 4)  # frio en trimestres centrales (invierno)
        temp = 24 - 0.0045 * altitud[r] + estacional + rng.normal(0, 0.8)
        cob_n = np.clip(cob_neumo_base[r] + 3.0 * (anio - 2023) + rng.normal(0, 2), 40, 98)
        cob_f = np.clip(cob_flu_base[r] + 3.5 * (anio - 2023) + rng.normal(0, 2), 30, 95)
        pobreza = np.clip(pobreza_base[r] - 1.2 * (anio - 2023) + rng.normal(0, 1.5), 5, 70)
        campana = int(anio == 2025 or (anio == 2024 and trim >= 2 and rng.random() < 0.6))
        # Relacion simulada (SUPUESTO): menos casos con mas cobertura/temperatura, mas con pobreza
        tasa = 92 * np.exp(
            -0.006 * (cob_n - 65) - 0.004 * (cob_f - 55)
            - 0.030 * (temp - 15) + 0.011 * (pobreza - 30) - 0.06 * campana
        )
        tasa *= np.exp(rng.normal(0, 0.05))
        filas.append([regiones[r], anio, trim + 1, cob_n, cob_f, temp, pobreza, campana, pob_menor5[r], tasa])

df = pd.DataFrame(filas, columns=["region", "anio", "trimestre", "cobertura_neumococo_pct",
                                  "cobertura_influenza_pct", "temperatura_media_c", "pobreza_pct",
                                  "campana_abrigamos_vidas", "pob_menor5", "tasa_ira_x1000"])

# Calibracion: escalar la tasa para que los totales anuales coincidan con el anexo
df["episodios"] = df["tasa_ira_x1000"] * df["pob_menor5"] / 1000
for a in anios:
    f = episodios_ira[a] / df.loc[df.anio == a, "episodios"].sum()
    df.loc[df.anio == a, "tasa_ira_x1000"] *= f
    print(f"Factor de calibracion {a}: {f:.3f}")
df["episodios"] = df["tasa_ira_x1000"] * df["pob_menor5"] / 1000

print("\nTotales anuales del dataset sintetico vs. anexo:")
print(df.groupby("anio")["episodios"].sum().round().astype(int).to_frame("sintetico").assign(anexo=pd.Series(episodios_ira)))
df.head()

In [ ]:
df.describe().round(2)

## 3. Exploración visual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
sc = axes[0].scatter(df["cobertura_neumococo_pct"], df["tasa_ira_x1000"], c=df["temperatura_media_c"], cmap="coolwarm_r", alpha=0.8)
axes[0].set_xlabel("Cobertura neumococo (%)"); axes[0].set_ylabel("Tasa IRA por 1000"); axes[0].set_title("Tasa vs. cobertura (color = temperatura)")
plt.colorbar(sc, ax=axes[0], label="°C")
axes[1].scatter(df["temperatura_media_c"], df["tasa_ira_x1000"], color="#0E7C7B", alpha=0.7)
axes[1].set_xlabel("Temperatura media (°C)"); axes[1].set_title("Tasa vs. temperatura")
axes[2].scatter(df["pobreza_pct"], df["tasa_ira_x1000"], color="#E8842C", alpha=0.7)
axes[2].set_xlabel("Pobreza (%)"); axes[2].set_title("Tasa vs. pobreza")
plt.tight_layout(); plt.show()

## 4. Modelo de regresión elegido y justificación

Se comparan dos modelos:

1. **Regresión lineal múltiple sobre `log(tasa)`** (modelo elegido). Es **interpretable**: cada coeficiente indica cuánto cambia la tasa (en %) por cada desviación estándar de una variable. Esto es clave en salud pública, donde importa *entender* qué factores se asocian con la tasa y no solo predecirla. Modelar `log(tasa)` es razonable porque los efectos son multiplicativos (por ejemplo, la vacunación reduce un % de los casos, no una cantidad fija) y la tasa es siempre positiva.
2. **Random Forest** (modelo de referencia, no lineal). Se usa como *benchmark*: si no mejora claramente a la regresión lineal, no se justifica sacrificar interpretabilidad.

La evaluación usa un conjunto de prueba (25 %) y validación cruzada de 5 particiones, con MAE, RMSE y R².

In [ ]:
features = ["cobertura_neumococo_pct", "cobertura_influenza_pct", "temperatura_media_c", "pobreza_pct", "campana_abrigamos_vidas"]
X = df[features]
y = np.log(df["tasa_ira_x1000"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

lineal = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(X_train, y_train)

def evaluar(nombre, modelo):
    pred = modelo.predict(X_test)
    y_real, y_pred = np.exp(y_test), np.exp(pred)    # volver a la escala original (tasa por 1000)
    cv = cross_val_score(modelo, X, y, cv=5, scoring="r2").mean()
    return {"Modelo": nombre, "MAE (x1000)": round(mean_absolute_error(y_real, y_pred), 2),
            "RMSE (x1000)": round(mean_squared_error(y_real, y_pred) ** 0.5, 2),
            "R2 test": round(r2_score(y_real, y_pred), 3), "R2 CV (5)": round(cv, 3)}

resultados = pd.DataFrame([evaluar("Regresion lineal (log)", lineal), evaluar("Random Forest", rf)])
resultados

## 5. Gráfico interpretativo

In [ ]:
pred_test = np.exp(lineal.predict(X_test)); real_test = np.exp(y_test)
coefs = lineal.named_steps["linearregression"].coef_
efecto_pct = (np.exp(coefs) - 1) * 100       # cambio % de la tasa por +1 desviacion estandar

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# (a) Predicho vs real
axes[0].scatter(real_test, pred_test, color="#0B3D5C", alpha=0.7)
lim = [min(real_test.min(), pred_test.min()), max(real_test.max(), pred_test.max())]
axes[0].plot(lim, lim, "--", color="#B0201B", label="Predicción perfecta")
axes[0].set_xlabel("Tasa real por 1000"); axes[0].set_ylabel("Tasa predicha por 1000")
axes[0].set_title(f"Predicho vs. real (R² = {resultados.loc[0, 'R2 test']})"); axes[0].legend()

# (b) Efecto de cada variable
orden = np.argsort(efecto_pct)
colores = ["#2E9E5B" if v < 0 else "#B0201B" for v in efecto_pct[orden]]
axes[1].barh(np.array(features)[orden], efecto_pct[orden], color=colores)
axes[1].axvline(0, color="#333", lw=0.8)
axes[1].set_xlabel("Cambio % de la tasa por +1 desv. estándar")
axes[1].set_title("¿Qué se asocia con más / menos IRA?")

# (c) Escenario: efecto de la cobertura de neumococo (resto en su media)
rango = np.linspace(df["cobertura_neumococo_pct"].min(), df["cobertura_neumococo_pct"].max(), 50)
esc = pd.DataFrame({f: df[f].mean() for f in features}, index=range(50))
esc["cobertura_neumococo_pct"] = rango
axes[2].scatter(df["cobertura_neumococo_pct"], df["tasa_ira_x1000"], color="#c9d2d8", alpha=0.6, label="Datos sintéticos")
axes[2].plot(rango, np.exp(lineal.predict(esc)), color="#E8842C", lw=3, label="Predicción del modelo")
axes[2].set_xlabel("Cobertura de vacuna neumococo (%)"); axes[2].set_ylabel("Tasa IRA por 1000")
axes[2].set_title("Escenario: más cobertura, menos casos"); axes[2].legend()
plt.tight_layout(); plt.show()

pd.DataFrame({"variable": features, "cambio % por +1 desv. est.": efecto_pct.round(2)}).sort_values("cambio % por +1 desv. est.")

## 6. Interpretación y conclusiones

- La tarea es de **regresión** porque la variable objetivo (tasa de episodios por 1 000 niños) es **continua**.
- El gráfico **Predicho vs. real** muestra puntos cerca de la diagonal, lo que indica buen ajuste en datos no vistos.
- El gráfico de **efectos** muestra que, en este dataset, mayor cobertura de vacunación y mayor temperatura se asocian con **menor** tasa, y mayor pobreza con **mayor** tasa. La campaña "Abrigamos vidas" aparece con un efecto negativo.
- El gráfico de **escenario** permite simular cuánto bajaría la tasa al subir la cobertura de neumococo, manteniendo el resto de variables en su promedio.
- La regresión lineal supera al Random Forest en el conjunto de prueba y en validación cruzada. Esto es esperable porque la simulación se programó con relaciones log-lineales; en datos reales el resultado podría cambiar. Se mantiene como modelo final por su **interpretabilidad**.

**Limitaciones:** (1) los datos son sintéticos y las relaciones fueron **programadas** en la simulación, así que los coeficientes reflejan esos supuestos y no evidencia real; (2) la infografía advierte que la **relación causal no está demostrada**, y una asociación del modelo no implica causalidad; (3) con datos regionales reales de cobertura, clima y pobreza (hoy no reportados) el modelo debería reentrenarse y validarse.